In [0]:
%run "./config"

In [0]:
(
    df_orders.write.format('delta')
    #initiating a write operation on df_orders, write as Delta format — Parquet files + _delta_log underneath
    .mode('Append')
    #adds new rows to existing data, never deletes old rows
    .option("path", "abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders")
    #target folder path
    .save()  #triggers the actual write — nothing runs until this line
)

## Write modes
1) overwrite — deletes all existing data in the table and writes fresh data. Every run starts from scratch.

2) append — adds new rows on top of existing data. Never touches old rows.

3) ignore — if the table or file already exists, skip the write entirely and do nothing. If it doesn't exist, write normally.

4) errorIfExists — if the table or file already exists, throw an error and stop. This is the default mode if you don't specify anything.


## Read modes
1) PERMISSIVE — reads all rows including bad/corrupt ones. Corrupt data goes into a separate column called _corrupt_record so you can inspect it later. Default mode.

2) DROPMALFORMED — reads only valid rows. Silently drops any row that doesn't match the schema or is corrupted. No error thrown.

3) FAILFAST — stops reading immediately the moment it hits a single bad row and throws an error. Zero tolerance for bad data.

In [0]:
df_orders.write.format('delta')\
    .mode('error')\
        .option('path','abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders')\
            .save()

#it throws error: [DELTA_PATH_EXISTS] Cannot write to already existent path abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders without setting OVERWRITE = 'true'



In [0]:
df_orders.write.format('delta')\
    .mode('ignore')\
        .option('path','abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders')\
            .save()

# if data already exists at this path — skip, do nothing
# if path is empty — write normally

In [0]:
df_orders.write.format('delta')\
    .mode('overwrite')\
        .option('path','abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders')\
            .save()

In [0]:
%sql
describe history delta.`abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders`

In [0]:
%sql
select * from delta.`abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders`

##Managed Delta Tables
a Delta table where Databricks controls both the metadata and the physical data files. Data is stored in Databricks internal storage. Dropping the table permanently deletes the data with no recovery option.

##External Delta Tables
a Delta table where you control the data location. Metadata lives in Databricks but the actual data files sit in your own Azure Data Lake Storage. Dropping the table only removes the metadata — data files in ADLS remain completely safe and can be re-registered anytime.

######Creating a Database

In [0]:
%sql
create database if not exists SalesDB

##### Managed Table

In [0]:
#creating a managed delta table from existing dataframe
if 'df_orders' not in locals():
    df_orders = spark.read.format('delta').load('abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders')

df_orders.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('SalesDB.drvd_orders')

In [0]:
%sql
select * from SalesDB.drvd_orders
limit 20

In [0]:
%sql
--creating new managed table under SalesDB
CREATE TABLE SalesDB.customers
(
    Id INT,
    Name STRING,
    Age INT
)
USING DELTA

In [0]:
%sql
select * from SalesDB.customers

In [0]:
%sql
insert into salesdb.customers values
    (1,"Raj",25),
    (2,"Kiran",30),
    (3,"Shiv",38)

######External Table

In [0]:

# write directly to path — bypasses Unity Catalog check
df_orders.write.format("delta")\
    .mode("overwrite")\
    .save("abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS SalesDB.orders_ext
USING DELTA
LOCATION 'abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders'

###### DATA VERSIONING

In [0]:

%sql
DESCRIBE HISTORY SalesDB.drvd_orders;

In [0]:
%sql
DESCRIBE HISTORY delta.`abfss://silver@storage4spotifyproject.dfs.core.windows.net/orders`

In [0]:
%sql
delete from SalesDB.customers
where Name = 'Raj'

In [0]:
%sql
SELECT * FROM salesdb.customers

In [0]:
%sql
describe history SalesDB.Customers

######Time Travel

In [0]:
%sql
RESTORE TABLE SalesDB.customers TO VERSION AS OF 1

######OPTIMIZATION

In [0]:
%sql
OPTIMIZE SalesDB.customers
--runtime decreased from 2.65s to 1.92s

######ZORDER BY

In [0]:
%sql
OPTIMIZE SalesDB.customers ZORDER BY(Name)

In [0]:
%sql
select * from salesdb.customers
--runtime decreased from 1.92s to1.65s

In [0]:
--